In [ ]:
from datetime import datetime
import openmeteo_requests

# Iterators

class IncreaseSpeed():
  def __init__(self, current_speed: int, max_speed: int, step=10):
    self.current_speed = current_speed
    self.max_speed = max_speed
    self.step = step

  def __iter__(self):
    return self

  def __next__(self):
    if self.current_speed >= self.max_speed:
      raise StopIteration

    self.current_speed += self.step
    if self.current_speed > self.max_speed:
      self.current_speed = self.max_speed

    print("INFO: Speed increases by 10")
    return self.current_speed


class DecreaseSpeed():
  def __init__(self, current_speed: int, min_speed: int, step=10):
    self.current_speed = current_speed
    self.min_speed = min_speed
    self.step = step

  def __iter__(self):
    return self

  def __next__(self):
    if self.current_speed <= self.min_speed:
      raise StopIteration

    self.current_speed -= self.step
    if self.current_speed < self.min_speed:
      self.current_speed = self.min_speed

    print("INFO: Speed decreases by 10")
    return self.current_speed

# Car

class Car():
  cars_on_road = 0

  def __init__(self, max_speed: int, current_speed=0):
    self.max_speed = max_speed
    self.current_speed = current_speed

    # if speed > 0 -> car is on road
    self.state = current_speed > 0

    if self.state:
      Car.cars_on_road += 1


  def accelerate(self, upper_border=None, step=10):
    start_speed = self.current_speed

    # if car was parked -> put on road
    if not self.state:
      self.state = True
      Car.cars_on_road += 1

    target = upper_border if upper_border is not None else self.max_speed
    target = min(target, self.max_speed)

    iterator = IncreaseSpeed(self.current_speed, target, step)

    for speed in iterator:
      self.current_speed = speed

    print(f"INFO: The speed of this car has been increased from {start_speed} to {self.current_speed}")


  def brake(self, lower_border=None, step=10):
    start_speed = self.current_speed

    target = lower_border if lower_border is not None else 0
    target = max(target, 0)

    iterator = DecreaseSpeed(self.current_speed, target, step)

    for speed in iterator:
      self.current_speed = speed

    print(f"INFO: The speed of this car has been decreased from {start_speed} to {self.current_speed}")


  def parking(self):
    
    self.brake(0)

    if not self.state:
      return

    print("Parking the car...")
    self.state = False
    Car.cars_on_road -= 1


  @classmethod
  def total_cars(cls):
    return cls.cars_on_road


  @staticmethod
  def show_weather():
    openmeteo = openmeteo_requests.Client()

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
      "latitude": 59.9386,
      "longitude": 30.3141,
      "current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"],
      "wind_speed_unit": "ms",
      "timezone": "Europe/Moscow"
    }

    response = openmeteo.weather_api(url, params=params)[0]

    current = response.Current()

    temp = current.Variables(0).Value()
    feels = current.Variables(1).Value()
    rain = current.Variables(2).Value()
    wind = current.Variables(3).Value()

    print(f"Current temperature: {round(temp, 0)} C")
    print(f"Current apparent_temperature: {round(feels, 0)} C")
    print(f"Current rain: {rain} mm")
    print(f"Current wind_speed: {round(wind, 1)} m/s")
